# 모델·도구·루프와 정지 조건

완성된 에이전트를 만들지 않고 사건 목록으로 루프와 정지 이유를 읽습니다.

## 실행 환경 준비

아래 코드는 모델이나 도구를 호출하지 않는 상태 전이 시뮬레이터입니다. Ch 20의 ToolNode 그래프 루프를 미리 구현하지 않습니다.

In [1]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
from pathlib import Path
import sys
from typing import TypedDict

start = Path.cwd().resolve()
candidates = (start, *start.parents)
ROOT = next(path for path in candidates if (path / "labs/part5/ch17-agent-overview").is_dir())
LAB = ROOT / "labs/part5/ch17-agent-overview"
print("Python:", sys.version.split()[0], "| 실습 폴더:", LAB.relative_to(ROOT))

Python: 3.12.13 | 실습 폴더: labs/part5/ch17-agent-overview


## 세 요소와 관찰

모델은 다음 행동을 제안하고, 도구는 환경에서 결과를 가져오며, 루프는 결과를 다음 판단에 다시 넣습니다.

In [2]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
class TraceEvent(TypedDict):
    step: int
    actor: str
    event: str
    progress: float
    cost: int

trace: list[TraceEvent] = [
    {"step": 1, "actor": "model", "event": "search", "progress": 0.20, "cost": 1},
    {"step": 2, "actor": "tool", "event": "search_result", "progress": 0.45, "cost": 0},
    {"step": 3, "actor": "model", "event": "read", "progress": 0.65, "cost": 1},
    {"step": 4, "actor": "tool", "event": "read_result", "progress": 0.90, "cost": 0},
    {"step": 5, "actor": "model", "event": "finish", "progress": 1.00, "cost": 1},
]
for event in trace:
    print(event)

{'step': 1, 'actor': 'model', 'event': 'search', 'progress': 0.2, 'cost': 1}
{'step': 2, 'actor': 'tool', 'event': 'search_result', 'progress': 0.45, 'cost': 0}
{'step': 3, 'actor': 'model', 'event': 'read', 'progress': 0.65, 'cost': 1}
{'step': 4, 'actor': 'tool', 'event': 'read_result', 'progress': 0.9, 'cost': 0}
{'step': 5, 'actor': 'model', 'event': 'finish', 'progress': 1.0, 'cost': 1}


## 정지 판정

완료, 최대 단계, 예산, 사람 확인 요청을 서로 다른 종료 이유로 보존합니다.

In [3]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
def stop_reason(
    events: list[TraceEvent], max_steps: int = 6, max_cost: int = 4
) -> str | None:
    if not events:
        return None
    if events[-1]["event"] == "finish":
        return "완료"
    if any(item["event"] == "human_check" for item in events):
        return "사람 확인"
    if len(events) >= max_steps:
        return "최대 단계"
    if sum(item["cost"] for item in events) >= max_cost:
        return "예산"
    return None

print("기본 추적 정지:", stop_reason(trace))

기본 추적 정지: 완료


## 완료 사건을 제거한 비교

마지막 완료를 제거하면 아직 실행 중인 상태로 판정됩니다.

In [4]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
unfinished: list[TraceEvent] = trace[:-1]
print("사건 수:", len(unfinished))
print("정지 이유:", stop_reason(unfinished))
assert stop_reason(unfinished) is None

사건 수: 4
정지 이유: None


## 세 가지 정지 사례

각 사례는 정지 조건의 우선순위를 확인하기 위한 교육용 사건입니다.

In [5]:
# 이 셀의 계산과 출력에서 현재 단계의 결과를 확인합니다.
maxed: list[TraceEvent] = unfinished + [{"step": 5, "actor": "model", "event": "retry", "progress": .9, "cost": 1},
                      {"step": 6, "actor": "tool", "event": "same_result", "progress": .9, "cost": 0}]
budgeted: list[TraceEvent] = trace[:4] + [{"step": 5, "actor": "model", "event": "retry", "progress": .9, "cost": 2}]
human: list[TraceEvent] = trace[:2] + [{"step": 3, "actor": "model", "event": "human_check", "progress": .45, "cost": 1}]
for name, events in [("반복 상한", maxed), ("예산 상한", budgeted), ("확인 요청", human)]:
    print(name, "→", stop_reason(events))
assert stop_reason(maxed) == "최대 단계"
assert stop_reason(budgeted) == "예산"
assert stop_reason(human) == "사람 확인"

반복 상한 → 최대 단계
예산 상한 → 예산
확인 요청 → 사람 확인
